In [2]:
import pandas as pd

# Creacion de la base de datos

In [21]:
comprobantes = pd.read_excel('Comprobantes detallados.xlsx', header=7)
comprobantes.head(10)

,Secuencia,Fecha elaboración,Código contable,Cuenta contable,Identificación,Sucursal,Nombre tercero,Descripción,Detalle,Centro de costo,Débito,Crédito
0,Comprobante: A-376,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,791634.31,791634.31
1,1,07/02/2026,14350101.0,Mercancías no fabricadas,901678785.0,0.0,PROVEEDORA ELECTRICA NACIONAL SAS,C. 12AWG CU ROJO THHN-2 600V ROLLO X 100,Prod: NX10011765 Cant: 1.00,NaN,0.00,199730.25
2,2,07/02/2026,61350520.0,Ajuste de Inventarió,901678785.0,0.0,PROVEEDORA ELECTRICA NACIONAL SAS,Ajuste de Inventarió,NaN,NaN,199730.25,0.00
3,3,07/02/2026,14350101.0,Mercancías no fabricadas,901678785.0,0.0,PROVEEDORA ELECTRICA NACIONAL SAS,C. 12AWG CU ROJO THHN-2 600V X MT,Prod: NX10024729 Cant: 100.00,NaN,200670.00,0.00
4,4,07/02/2026,61350520.0,Ajuste de Inventarió,901678785.0,0.0,PROVEEDORA ELECTRICA NACIONAL SAS,Ajuste de Inventarió,NaN,NaN,0.00,200670.00
5,5,07/02/2026,14350101.0,Mercancías no fabricadas,901678785.0,0.0,PROVEEDORA ELECTRICA NACIONAL SAS,C. 12AWG CU VERDE THHN-2 600V ROLLO X 100,Prod: NX10011767 Cant: 1.00,NaN,0.00,199072.06
6,6,07/02/2026,61350520.0,Ajuste de Inventarió,901678785.0,0.0,PROVEEDORA ELECTRICA NACIONAL SAS,Ajuste de Inventarió,NaN,NaN,199072.06,0.00
7,7,07/02/2026,14350101.0,Mercancías no fabricadas,901678785.0,0.0,PROVEEDORA ELECTRICA NACIONAL SAS,C. 12AWG CU VERDE THHN-2 600V X MT,Prod: NX10024731 Cant: 100.00,NaN,192162.00,0.00
8,8,07/02/2026,61350520.0,Ajuste de Inventarió,901678785.0,0.0,PROVEEDORA ELECTRICA NACIONAL SAS,Ajuste de Inventarió,NaN,NaN,0.00,192162.00
9,Comprobante: A-377,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1396.07,1396.07


In [22]:
# Extraer código de comprobante de las filas encabezado y propagarlo
comprobantes['Comprobante'] = comprobantes['Secuencia'].where(
    comprobantes['Secuencia'].astype(str).str.startswith('Comprobante:')
).ffill()

# Extraer solo el código (ej: "A-376" de "Comprobante: A-376")
comprobantes['Comprobante'] = comprobantes['Comprobante'].str.replace('Comprobante: ', '', regex=False)

# Eliminar las filas encabezado de comprobante (quedan solo las secuencias)
comprobantes = comprobantes[~comprobantes['Secuencia'].astype(str).str.startswith('Comprobante:')].reset_index(drop=True)

# Convertir columnas a entero (Int64 soporta NaN)
comprobantes['Código contable'] = comprobantes['Código contable'].astype('Int64')
comprobantes['Identificación'] = comprobantes['Identificación'].astype('Int64')
comprobantes['Sucursal'] = comprobantes['Sucursal'].astype('Int64')

# Crear columna Movimiento: crédito cuando débito es 0, -débito cuando crédito es 0
comprobantes['Movimiento'] = comprobantes.apply(
    lambda r: r['Crédito'] if r['Débito'] == 0 else -r['Débito'], axis=1
)

# Mover Comprobante al inicio del dataframe
cols = ['Comprobante'] + [c for c in comprobantes.columns if c != 'Comprobante']
comprobantes = comprobantes[cols]

comprobantes.head(20)

,Comprobante,Secuencia,Fecha elaboración,Código contable,Cuenta contable,Identificación,Sucursal,Nombre tercero,Descripción,Detalle,Centro de costo,Débito,Crédito,Movimiento
0,A-376,1,07/02/2026,14350101,Mercancías no fabricadas,901678785,0,PROVEEDORA ELECTRICA NACIONAL SAS,C. 12AWG CU ROJO THHN-2 600V ROLLO X 100,Prod: NX10011765 Cant: 1.00,NaN,0.00,199730.25,199730.25
1,A-376,2,07/02/2026,61350520,Ajuste de Inventarió,901678785,0,PROVEEDORA ELECTRICA NACIONAL SAS,Ajuste de Inventarió,NaN,NaN,199730.25,0.00,-199730.25
2,A-376,3,07/02/2026,14350101,Mercancías no fabricadas,901678785,0,PROVEEDORA ELECTRICA NACIONAL SAS,C. 12AWG CU ROJO THHN-2 600V X MT,Prod: NX10024729 Cant: 100.00,NaN,200670.00,0.00,-200670.00
3,A-376,4,07/02/2026,61350520,Ajuste de Inventarió,901678785,0,PROVEEDORA ELECTRICA NACIONAL SAS,Ajuste de Inventarió,NaN,NaN,0.00,200670.00,200670.00
4,A-376,5,07/02/2026,14350101,Mercancías no fabricadas,901678785,0,PROVEEDORA ELECTRICA NACIONAL SAS,C. 12AWG CU VERDE THHN-2 600V ROLLO X 100,Prod: NX10011767 Cant: 1.00,NaN,0.00,199072.06,199072.06
5,A-376,6,07/02/2026,61350520,Ajuste de Inventarió,901678785,0,PROVEEDORA ELECTRICA NACIONAL SAS,Ajuste de Inventarió,NaN,NaN,199072.06,0.00,-199072.06
6,A-376,7,07/02/2026,14350101,Mercancías no fabricadas,901678785,0,PROVEEDORA ELECTRICA NACIONAL SAS,C. 12AWG CU VERDE THHN-2 600V X MT,Prod: NX10024731 Cant: 100.00,NaN,192162.00,0.00,-192162.00
7,A-376,8,07/02/2026,61350520,Ajuste de Inventarió,901678785,0,PROVEEDORA ELECTRICA NACIONAL SAS,Ajuste de Inventarió,NaN,NaN,0.00,192162.00,192162.00
8,A-377,1,09/02/2026,14350101,Mercancías no fabricadas,901678785,0,PROVEEDORA ELECTRICA NACIONAL SAS,CINTA ALISLANTE NEGRA 15 MTS GRANDE UL FMX VL ...,Prod: KL-1227 Cant: 0.00,NaN,0.00,1396.07,1396.07
9,A-377,2,09/02/2026,61350515,"Retiro de inventario para activos fijos, consu...",901678785,0,PROVEEDORA ELECTRICA NACIONAL SAS,"Retiro de inventario para activos fijos, consu...",NaN,NaN,1396.07,0.00,-1396.07


In [23]:
# guardar comprobantes en un archivo csv
comprobantes.to_csv('comprobantes.csv', index=False)

In [24]:
productos = pd.read_excel('Lista de productos.xlsx')
productos.head(10)

,Código producto,Nombre producto,MARCA DETALLADA,MARCA RESUMIDA,LINEA NEGOCIO
0,000001EM,CAJA MECANISMO 32MM ECO. EM,EM,SECUNDARIA,OTROS
1,077DEX004,CANALETA RANURADA GRIS 40*40 2M E,OTROS,SECUNDARIA,OTROS
2,10100B,BOMBILLO LED GU10 5W 6500K WELLMAX,WELLMAX,SECUNDARIA,ILUMINACION
3,10100C,BOMBILLO LED GU10 5W 3500K CEB,CEB,SECUNDARIA,ILUMINACION
4,10101B,BOMBILLO LED GU10 EDICION 12 6500K WELLMAX,WELLMAX,SECUNDARIA,ILUMINACION
5,10103B,BOMBILLO LED 5W 6500K WELLMAX,WELLMAX,SECUNDARIA,ILUMINACION
6,10105B,BOMBILLO LED 12W 6500K WELLMAX,WELLMAX,SECUNDARIA,ILUMINACION
7,10106B,BOMBILLO LED 15W 6500K WELLMAX,WELLMAX,SECUNDARIA,ILUMINACION
8,10112B,PANEL LED REDONDO INC 3W 6500K WELLMAX,WELLMAX,SECUNDARIA,ILUMINACION
9,10136,CHASIS OJO DE BUEY CEB,CEB,SECUNDARIA,ILUMINACION


In [31]:
# Extraer código de producto desde Detalle (ej: "Prod: NX10011765 Cant: 1.00")
comprobantes['Código producto'] = comprobantes['Detalle'].str.extract(r'Prod:\s*(\S+)')

# Merge con productos para traer info del producto
comprobantes = comprobantes.merge(productos[['Código producto', 'MARCA RESUMIDA', 'LINEA NEGOCIO']], on='Código producto', how='left')

comprobantes.head(20)

,Comprobante,Secuencia,Fecha elaboración,Código contable,Cuenta contable,Identificación,Sucursal,Nombre tercero,Descripción,Detalle,Centro de costo,Débito,Crédito,Movimiento,Código producto,MARCA RESUMIDA,LINEA NEGOCIO
0,A-376,1,07/02/2026,14350101,Mercancías no fabricadas,901678785,0,PROVEEDORA ELECTRICA NACIONAL SAS,C. 12AWG CU ROJO THHN-2 600V ROLLO X 100,Prod: NX10011765 Cant: 1.00,NaN,0.00,199730.25,199730.25,NX10011765,PRIMARIA,CABLEADO
1,A-376,2,07/02/2026,61350520,Ajuste de Inventarió,901678785,0,PROVEEDORA ELECTRICA NACIONAL SAS,Ajuste de Inventarió,NaN,NaN,199730.25,0.00,-199730.25,NaN,NaN,NaN
2,A-376,3,07/02/2026,14350101,Mercancías no fabricadas,901678785,0,PROVEEDORA ELECTRICA NACIONAL SAS,C. 12AWG CU ROJO THHN-2 600V X MT,Prod: NX10024729 Cant: 100.00,NaN,200670.00,0.00,-200670.00,NX10024729,PRIMARIA,CABLEADO
3,A-376,4,07/02/2026,61350520,Ajuste de Inventarió,901678785,0,PROVEEDORA ELECTRICA NACIONAL SAS,Ajuste de Inventarió,NaN,NaN,0.00,200670.00,200670.00,NaN,NaN,NaN
4,A-376,5,07/02/2026,14350101,Mercancías no fabricadas,901678785,0,PROVEEDORA ELECTRICA NACIONAL SAS,C. 12AWG CU VERDE THHN-2 600V ROLLO X 100,Prod: NX10011767 Cant: 1.00,NaN,0.00,199072.06,199072.06,NX10011767,PRIMARIA,CABLEADO
5,A-376,6,07/02/2026,61350520,Ajuste de Inventarió,901678785,0,PROVEEDORA ELECTRICA NACIONAL SAS,Ajuste de Inventarió,NaN,NaN,199072.06,0.00,-199072.06,NaN,NaN,NaN
6,A-376,7,07/02/2026,14350101,Mercancías no fabricadas,901678785,0,PROVEEDORA ELECTRICA NACIONAL SAS,C. 12AWG CU VERDE THHN-2 600V X MT,Prod: NX10024731 Cant: 100.00,NaN,192162.00,0.00,-192162.00,NX10024731,PRIMARIA,CABLEADO
7,A-376,8,07/02/2026,61350520,Ajuste de Inventarió,901678785,0,PROVEEDORA ELECTRICA NACIONAL SAS,Ajuste de Inventarió,NaN,NaN,0.00,192162.00,192162.00,NaN,NaN,NaN
8,A-377,1,09/02/2026,14350101,Mercancías no fabricadas,901678785,0,PROVEEDORA ELECTRICA NACIONAL SAS,CINTA ALISLANTE NEGRA 15 MTS GRANDE UL FMX VL ...,Prod: KL-1227 Cant: 0.00,NaN,0.00,1396.07,1396.07,KL-1227,SECUNDARIA,ILUMINACION
9,A-377,2,09/02/2026,61350515,"Retiro de inventario para activos fijos, consu...",901678785,0,PROVEEDORA ELECTRICA NACIONAL SAS,"Retiro de inventario para activos fijos, consu...",NaN,NaN,1396.07,0.00,-1396.07,NaN,NaN,NaN


In [32]:
# guardar productos en un archivo csv
productos.to_csv('productos.csv', index=False)

# Procesamiento de datos

In [33]:
comprobantes['Código contable'].unique()

<IntegerArray>
[  14350101,   61350520,   61350515,   22050501,   11100504,   13652002,
   11050503,   11050501,   13551501,   28050501,   11050502,   13551802,
 2510100101,   11100501,   51401597,     229999,   24081001,   23654001,
   51350502,   24081005,   51953002,   51952501,   24081003,   51953001,
   41350101,   24080601,   13551801,   41800101,   24080603,   14980101,
   41750501,   24082001,   13050501,   42104001,   42958101,   23689005,
       <NA>]
Length: 37, dtype: Int64